In [0]:
catalog = "workspace"

dbName = "ecommerceNew"

volume_name = "ecommerce_Deltadata"

In [0]:
spark.sql(f"Create schema if not exists {catalog}.{dbName}")


DataFrame[]

In [0]:
spark.sql(f"use {catalog}.{dbName}")

DataFrame[]

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(2)


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|53

In [0]:
df.write.format('delta').mode("overwrite").saveAsTable(f"{volume_name}")

In [0]:
edf = spark.sql(f"DESCRIBE DETAIL {volume_name}");
edf.count()

1

In [0]:
spark.sql(f"DESCRIBE history {volume_name}")

DataFrame[version: bigint, timestamp: timestamp, userId: string, userName: string, operation: string, operationParameters: map<string,string>, job: struct<jobId:string,jobName:string,jobRunId:string,runId:string,jobOwnerId:string,triggerType:string>, notebook: struct<notebookId:string>, clusterId: string, readVersion: bigint, isolationLevel: string, isBlindAppend: boolean, operationMetrics: map<string,string>, userMetadata: string, engineInfo: string]

In [0]:
old_df = spark.read.format("delta") \
    .option("timestampAsOf", "2026-01-13 20:40:30") \
    .table(f"{volume_name}")


In [0]:
old_df.show(2)

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|530496790|8e5f4f83-366c-4f7...|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
only showing top 2 rows


In [0]:
updates_df = df.limit(1000) \
    .withColumn("price", df.price * 1.1)


In [0]:
updates_df.show(2)

+-------------------+----------+----------+-------------------+--------------------+------+-----------------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand|            price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+-----------------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|537.9770000000001|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|          323.015|530496790|8e5f4f83-366c-4f7...|
+-------------------+----------+----------+-------------------+--------------------+------+-----------------+---------+--------------------+
only showing top 2 rows


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, f"{volume_name}")

delta_table.alias("t").merge(
    updates_df.alias("s"),
    "t.user_session = s.user_session AND t.event_time = s.event_time"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"OPTIMIZE {volume_name} ZORDER BY (user_id)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql(f"VACUUM {volume_name} RETAIN 168 HOURS;")

DataFrame[path: string]

In [0]:
fdf = spark.sql(f"SELECT COUNT(*) FROM {volume_name}")
fdf.show()

+--------+
|COUNT(*)|
+--------+
|67501979|
+--------+

